In [28]:
import dpluspy
import demes 
import moments
import msprime
import numpy as np
import pandas
import pickle
import sys

In [29]:
# Define models and parameters
model0_file = "models/graph0.yaml"
params0_file = "models/params0.yaml"
model1_file = "models/graph1.yaml"
params1_file = "models/params1.yaml"


# Parameters
r = 1e-8
u = 1.3e-8
u_std = 1e-9
L = int(5e6)
n_reps = 100
pop_ids = ["popX"]
samples = {pop_id: 1 for pop_id in pop_ids}
bins = np.logspace(-6, -2, 17)

In [30]:
def simulate():
    graph = demes.load(model1_file)
    demog = msprime.Demography.from_demes(graph)
    tss = msprime.sim_ancestry(
        demography=demog,
        samples=samples,
        sequence_length=L,
        recombination_rate=r,
        num_replicates=n_reps,
    )
    mtss = [msprime.sim_mutations(ts, rate=np.random.normal(u, u_std)) 
        for ts in tss]
    return mtss


def parse_stats(mtss):
    # Parse statistics from simulation results
    intervals = [[1, L + 1, L + 1]]
    pop_mapping = {x: [x] for x in samples} 
    stats = dict()
    for ii, ts in enumerate(mtss):
        if ii == 0:
            stats_ii = dpluspy.parsing.parse_stats(
                ts, 
                get_denoms=True,
                pop_mapping=pop_mapping, 
                r=r, 
                r_bins=bins, 
                intervals=intervals,
                chrom=ii, 
                ts_sample_ids=pop_ids, 
                overhang=None
            )
        else:
            stats_ii = dpluspy.parsing.parse_stats(
                ts, 
                get_denoms=False,
                pop_mapping=pop_mapping, 
                r=r, 
                r_bins=bins, 
                intervals=intervals,
                chrom=ii, 
                ts_sample_ids=pop_ids, 
                overhang=None
            )
            for jj in range(len(intervals)):
                stats_ii[(ii, jj)]["denoms"] = stats[(0, jj)]["denoms"]
        stats.update(stats_ii)
    return stats

[2025-07-08 17:11:03] Computed stats in chrom 0 interval 0
[2025-07-08 17:11:03] Computed stats in chrom 1 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 2 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 3 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 4 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 5 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 6 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 7 interval 0
[2025-07-08 17:11:04] Computed stats in chrom 8 interval 0
[2025-07-08 17:11:05] Computed stats in chrom 9 interval 0
[2025-07-08 17:11:05] Computed stats in chrom 10 interval 0
[2025-07-08 17:11:05] Computed stats in chrom 11 interval 0
[2025-07-08 17:11:05] Computed stats in chrom 12 interval 0
[2025-07-08 17:11:05] Computed stats in chrom 13 interval 0
[2025-07-08 17:11:05] Computed stats in chrom 14 interval 0
[2025-07-08 17:11:06] Computed stats in chrom 15 interval 0
[2025-07-08 17:11:06] Computed stats in chrom 16 i

In [ ]:
mtss = simulate()

In [ ]:
stats = parse_stats(mtss)

replicates = dpluspy.bootstrapping.get_bootstrap_reps(
    stats, num_reps=100, weighted=False)
varcovs = dpluspy.bootstrapping.compute_varcovs(replicates)
means = dpluspy.bootstrapping.means_across_regions(stats)
boot_stats = {"means": means, "varcovs": varcovs, "replicates": replicates,
    "bins": bins, "pop_ids": pop_ids}

In [31]:
def fit_model(graph_file, param_file):
    """
    Fit a model in two stages; return fitted parameters without writing output
    to disk
    """
    dpluspy.inference.optimize(
        graph_file,
        param_file,
        means,
        varcovs,
        pop_ids=pop_ids,
        bins=bins,
        u=u,
        perturb=0.2,
        method="lbfgsb",
        log=True,
        max_iter=10000,
        verbose=100,
        output="intermediate.yaml",
        overwrite=True
    )
    ret = dpluspy.inference.optimize(
        "intermediate.yaml",
        param_file,
        means,
        varcovs,
        pop_ids=pop_ids,
        bins=bins,
        u=u,
        method="powell",
        log=True,
        max_iter=100,
        verbose=100
    )
    return ret

In [33]:
# Fit the underlying and complex models; try to force ll1 > ll0
pnames0, params0, ll0 = fit_model(model0_file, params0_file)
ll1 = -1e10
max_tries = 20
tries = 0
while ll1 <= ll0:
    pnames1, params1, ll1 = fit_model(model1_file, params1_file)
    tries += 1
    if tries > max_tries:
        raise ValueError("Cannot achieve ll1 > ll0")

# We have one nested parameter, which is not at a boundary
weights = (0.5, 0.5)
D_naive = 2 * (ll1 - ll0)
p_naive = moments.Godambe.sum_chi2_ppf(D_naive, weights=weights)

# Set up model arguments
_, __, model_args = dpluspy.uncerts.set_up_model_args(
    model1_file, params1_file, bins=bins, pop_ids=pop_ids, u=u)


[08-07-25 17:11:54] Fitting D+ to data for ['popX']
Call         LL [       N_0]
init          - [  1.55e+04]
Finished with flag 2
Log-likelihood:	-56.5
Fitted parameters:
N_0	1.7e+04
[08-07-25 17:11:55] Fitting D+ to data for ['popX']
Call         LL [       N_0]
init          - [   1.7e+04]
Finished with flag 0
Log-likelihood:	-56.48
Fitted parameters:
N_0	1.7e+04
[08-07-25 17:11:55] Fitting D+ to data for ['popX']
Call         LL [       N_0     T_EXP       N_1]
init          - [  1.41e+04  6.49e+03  1.05e+05]
100       -0.26 [  1.49e+04  6.47e+03  1.07e+05]
Finished with flag 0
Log-likelihood:	-0.26
Fitted parameters:
N_0	1.49e+04
T_EXP	6.47e+03
N_1	1.07e+05
[08-07-25 17:12:00] Fitting D+ to data for ['popX']
Call         LL [       N_0     T_EXP       N_1]
init          - [  1.49e+04  6.47e+03  1.07e+05]
100       -3.94 [   1.5e+04  6.33e+03  1.48e+05]
Finished with flag 0
Log-likelihood:	-0.06
Fitted parameters:
N_0	1.5e+04
T_EXP	6.21e+03
N_1	9.83e+04


In [34]:
print(ll0)
print(ll1)
print(D_naive)

-56.48022048571174
-0.058661268764883784
112.84311843389371


In [46]:
# Compute LRT adjustment at the simple model MLE parameter values:
p0 = np.array([params0[0], params1[1], params0[0]])
steps = np.array([3000, params1[2]]) * 0.01
nested_idx = np.array([1, 2])

print(p0, steps)

adj_simple = dpluspy.uncerts.LRT_adjust(
    p0, 
    model_args, 
    means, 
    varcovs, 
    replicates, 
    nested_idx, 
    verbose=True,
    steps=steps
)

# Compute the LR test statistic
D_simple = adj_simple * D_naive
p_simple = moments.Godambe.sum_chi2_ppf(D_simple, weights=weights)

print(adj_simple, D_simple, p_simple)

[16988.46520458  6213.98937491 16988.46520458] [ 30.         982.84868729]
[08-07-25 17:22:32] Evaluated Hessian element (0, 0)
[08-07-25 17:22:32] Evaluated Hessian element (0, 1)
[08-07-25 17:22:32] Evaluated Hessian element (1, 1)
161599440.87057006 18235384845.00874 0.0


In [42]:
# Now compute LRT adjustment at the complex model MLE parameter values,
p1 = params1
steps = params1[1:] * 0.01

print(p1, steps)

adj_complex = dpluspy.uncerts.LRT_adjust(
    p1, 
    model_args, 
    means, 
    varcovs, 
    replicates, 
    nested_idx, 
    verbose=True,
    steps=steps
)

D_complex = adj_complex * D_naive
p_complex = moments.Godambe.sum_chi2_ppf(D_complex, weights=weights)

print(adj_complex, D_complex, p_complex)

[15042.28094213  6213.98937491 98284.86872934] [ 62.13989375 982.84868729]
[08-07-25 17:18:07] Evaluated Hessian element (0, 0)
[08-07-25 17:18:07] Evaluated Hessian element (0, 1)
[08-07-25 17:18:07] Evaluated Hessian element (1, 1)
0.13210137896799917 14.906731552166606 5.647959627197441e-05


In [ ]:
# TODO plots of LL space